# 04 — Baseline Model Training

**Research**: Analisis Performa Algoritma Machine Learning untuk Klasifikasi Jenis Cyberbullying pada Teks Bahasa Indonesia Menggunakan TF-IDF

**Objective**: Train baseline models with **default parameters** to establish a research baseline.

**Models**:
1. Multinomial Naive Bayes
2. Logistic Regression
3. Support Vector Machine (Linear)

**Rules**:
- Default parameters only — no hyperparameter tuning
- No evaluation metrics — evaluation is in a separate notebook

---

## 1. Setup

In [1]:
import sys
import os

# Ensure project root is in path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Import project modules
from src.config.settings import TRAINING_REPORTS_DIR
from src.config.model_config import ModelConfig
from src.utils.load_features import load_features
from src.training.model_factory import create_model
from src.training.model_registry import get_all_model_names, get_model_save_path
from src.training.trainer import (
    train_model,
    generate_training_report,
    export_training_statistics,
    save_training_report,
)
from src.training.persistence import save_model

print('Setup complete.')
print(f'Available models: {get_all_model_names()}')

Setup complete.
Available models: ['naive_bayes', 'logistic_regression', 'svm']


---
## 2. Load Features & Labels

In [2]:
X_train, X_test, y_train, y_test = load_features()

print(f'\nTraining set:  X={X_train.shape}, y={len(y_train):,}')
print(f'Testing set:   X={X_test.shape}, y={len(y_test):,}')
print(f'Labels: {sorted(y_train.unique())}')

✓ Artifacts loaded: X_train=(33244, 22695), X_test=(8312, 22695), vocab=22,695
✓ Features validated: X_train=(33244, 22695), X_test=(8312, 22695), labels=6 classes

Training set:  X=(33244, 22695), y=33,244
Testing set:   X=(8312, 22695), y=8,312
Labels: ['harassment', 'hate_speech', 'insult', 'normal', 'sexually_explicit', 'threat']


---
## 3. Configure Training

In [3]:
config = ModelConfig(
    random_state=42,
    overwrite=True,
    save_model=True,
    verbose=True,
)

print('Training Configuration:')
for k, v in config.to_dict().items():
    print(f'  {k}: {v}')

Training Configuration:
  random_state: 42
  model_output_directory: /home/zapp/Kampus/PM/models
  overwrite: True
  save_model: True
  verbose: True


---
## 4. Train Naive Bayes

In [4]:
nb_model = create_model('naive_bayes', config)
train_model(nb_model, X_train, y_train)
save_model(nb_model, get_model_save_path('naive_bayes'), overwrite=config.overwrite)

  Training NaiveBayes... done (0.08s)
  ✓ Model saved: /home/zapp/Kampus/PM/models/naive_bayes_baseline.joblib (2.1 MB)


'/home/zapp/Kampus/PM/models/naive_bayes_baseline.joblib'

---
## 5. Train Logistic Regression

In [5]:
lr_model = create_model('logistic_regression', config)
train_model(lr_model, X_train, y_train)
save_model(lr_model, get_model_save_path('logistic_regression'), overwrite=config.overwrite)

  Training LogisticRegression... done (13.12s)
  ✓ Model saved: /home/zapp/Kampus/PM/models/logistic_regression_baseline.joblib (1.0 MB)


'/home/zapp/Kampus/PM/models/logistic_regression_baseline.joblib'

---
## 6. Train SVM

⚠️ **Note**: SVM with `probability=True` uses Platt scaling, which adds training time.

In [6]:
svm_model = create_model('svm', config)
train_model(svm_model, X_train, y_train)
save_model(svm_model, get_model_save_path('svm'), overwrite=config.overwrite)

  Training SVM... done (526.28s)
  ✓ Model saved: /home/zapp/Kampus/PM/models/svm_baseline.joblib (4.7 MB)


'/home/zapp/Kampus/PM/models/svm_baseline.joblib'

---
## 7. Training Summary

In [7]:
trained_models = [nb_model, lr_model, svm_model]

print('Baseline Training Summary')
print('=' * 60)
print(f'{"Model":<25} {"Time (s)":<15} {"Status"}')
print('-' * 60)
for model in trained_models:
    meta = model.get_metadata()
    status = '✓ Trained' if meta['is_trained'] else '✗ Failed'
    print(f'{meta["model_name"]:<25} {meta["training_time_seconds"]:<15.4f} {status}')
print('=' * 60)

Baseline Training Summary
Model                     Time (s)        Status
------------------------------------------------------------
NaiveBayes                0.0829          ✓ Trained
LogisticRegression        13.1168         ✓ Trained
SVM                       526.2801        ✓ Trained


---
## 8. Generate Reports

In [8]:
print('Generating reports...\n')

# Training summary report (markdown)
report_content = generate_training_report(
    trained_models=trained_models,
    n_samples=X_train.shape[0],
    n_features=X_train.shape[1],
    config=config,
)
save_training_report(
    report_content,
    os.path.join(TRAINING_REPORTS_DIR, 'training_summary.md'),
)

# Training statistics CSV
export_training_statistics(
    trained_models=trained_models,
    n_samples=X_train.shape[0],
    n_features=X_train.shape[1],
    filepath=os.path.join(TRAINING_REPORTS_DIR, 'training_statistics.csv'),
)

print(f'\n✓ All reports generated.')

Generating reports...

  ✓ Training report saved: /home/zapp/Kampus/PM/reports/training/training_summary.md
  ✓ Training statistics exported: /home/zapp/Kampus/PM/reports/training/training_statistics.csv

✓ All reports generated.


---
## Summary

Baseline model training is complete. Generated outputs:

**Models**:
- `models/naive_bayes_baseline.joblib`
- `models/logistic_regression_baseline.joblib`
- `models/svm_baseline.joblib`

**Reports**:
- `reports/training/training_summary.md`
- `reports/training/training_statistics.csv`

**Next Step**: `05_hyperparameter_tuning.ipynb` — Hyperparameter Tuning